In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import datetime
import time
import random
import os

# --- CONFIGURATION ---
# Always check robots.txt before scraping! 
# E.g., https://www.jumia.ug/robots.txt
URL = https://www.jumia.ug
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.5",
}
CSV_FILE = "price_history.csv"

def get_product_details(url):
    """Fetches page content and extracts product title and price."""
    try:
        # Be polite: sleep between 2 to 5 seconds
        time.sleep(random.uniform(2, 5))
        
        response = requests.get(url, headers=HEADERS)
        # Raise an exception if the status code is not 200 (OK)
        response.raise_for_status() 
        
        soup = BeautifulSoup(response.content, "html.parser")

        # --- THESE SELECTORS ARE HYPOTHETICAL ---
        # YOU MUST INSPECT THE PAGE YOURSELF TO FIND THE RIGHT ONES
        # Right-click the price on the page -> Inspect
        try:
            title = soup.find(id="productTitle").get_text().strip()
        except AttributeError:
            title = "Unknown Product"
            
        try:
            # Example: Amazon prices are often inside spans with these classes
            price_whole = soup.find("span", {"class": "a-price-whole"}).get_text().strip()
            price_fraction = soup.find("span", {"class": "a-price-fraction"}).get_text().strip()
            price_string = f"{price_whole}{price_fraction}"
            # Clean up string (remove symbols like $)
            price = float(price_string.replace(",", "").replace("$", ""))
        except AttributeError:
            price = None # Could not find price

        return title, price

    except Exception as e:
        print(f"Error scraping {url}: {e}")
        return None, None

def log_price(title, price):
    """Appends the current price to a CSV file."""
    if price is None:
        print("Skipping log due to missing price.")
        return

    today = datetime.date.today().isoformat()
    data = {'Date': [today], 'Price': [price]}
    df = pd.DataFrame(data)

    # If file doesn't exist, create it with headers. If it exists, append without headers.
    header = not os.path.exists(CSV_FILE)
    
    try:
        df.to_csv(CSV_FILE, mode='a', header=header, index=False)
        print(f"Logged {today}: {price} for {title}")
    except Exception as e:
        print(f"Error writing to CSV: {e}")

# --- MAIN EXECUTION ---
if __name__ == "__main__":
    print("Starting scraper...")
    product_title, current_price = get_product_details(URL)
    
    if product_title and current_price:
        print(f"Product: {product_title}")
        print(f"Current Price: {current_price}")
        log_price(product_title, current_price)
    else:
        print("Failed to retrieve product details. Check URL or selectors.")

SyntaxError: invalid syntax (110904556.py, line 12)